# Project 5 — LangGraph Research Agent (Starter)

**Goal:** Build a multi-node LangGraph agent that researches a topic and produces a structured report.

**Stack:** LangGraph · OpenAI · LangChain

**Exercise cells are marked with `# YOUR CODE HERE`.**

---

**Graph architecture:**
```
START → planner → researcher (parallel) → writer → editor → END
                                                       ↑
                                            (loop back if quality < 0.8)
```

**Nodes to implement:**
1. `planner` — decomposes topic into 3-5 search queries
2. `researcher` — searches for information (stub provided)
3. `writer` — synthesizes findings into a report
4. `editor` — scores quality and loops back if needed

In [ ]:
%pip install -q langgraph langchain-openai langchain

In [ ]:
# Setup — do not modify
import os, json
from typing import TypedDict, Annotated, Optional
import operator
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3, api_key=os.getenv("OPENAI_API_KEY"))
llm_json = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=os.getenv("OPENAI_API_KEY"))

def chat(prompt: str, temperature: float = 0.3) -> str:
    r = llm.invoke(prompt)
    return r.content

# Stub search function — replace with Tavily, SerpAPI, or DuckDuckGo in production
def web_search(query: str) -> str:
    return (
        f"Search results for '{query}':\n"
        f"1. AI adoption grew 37% in 2023 according to McKinsey Global Survey.\n"
        f"2. GPT-4 achieved human-level performance on multiple professional exams.\n"
        f"3. 77% of devices now use AI in some form (Forbes, 2024).\n"
        f"4. EU AI Act passed in 2024, requiring transparency for high-risk AI systems.\n"
    )

print("Setup complete")

## Step 1 · Define State

In [ ]:
# Exercise 1: Define the graph state
# Required fields:
# - topic: str (the research topic)
# - queries: list[str] (search queries from planner)
# - findings: Annotated[list[str], operator.add] (accumulated from researchers)
# - report: str (written by the writer)
# - quality_score: float (given by editor, 0.0-1.0)
# - attempts: int (number of writer attempts)
# - feedback: str (editor feedback for writer)

# YOUR CODE HERE
class ResearchState(TypedDict):
    topic: str
    # Add remaining fields...

print("ResearchState defined")

## Step 2 · Implement Nodes

In [ ]:
# Exercise 2a: Implement the planner node
# Input: state["topic"]
# Output: {"queries": [list of 3-5 search queries]}
# Hint: ask the LLM to generate queries as a numbered list, then parse them

def planner(state: ResearchState) -> dict:
    # YOUR CODE HERE
    return {"queries": []}

# Test planner
test_state = {"topic": "impact of AI on healthcare", "queries": [], "findings": [], "report": "", "quality_score": 0.0, "attempts": 0, "feedback": ""}
result = planner(test_state)
print(f"Queries generated: {len(result['queries'])}")
for q in result["queries"]:
    print(f"  - {q}")

In [ ]:
# Exercise 2b: Implement the researcher node
# It should search ALL queries in state["queries"] (using web_search)
# Return {"findings": [list of search results]}
# Each finding should include the query and the result

def researcher(state: ResearchState) -> dict:
    # YOUR CODE HERE
    return {"findings": []}

# Test researcher
test_state["queries"] = ["AI in medical diagnosis", "AI drug discovery"]
result = researcher(test_state)
print(f"Findings: {len(result['findings'])}")
if result["findings"]:
    print(f"First finding: {result['findings'][0][:100]}...")

In [ ]:
# Exercise 2c: Implement the writer node
# Input: state["topic"], state["findings"], state["feedback"] (from previous editor run)
# Output: {"report": structured report string}
# The report should have: Introduction, Key Findings (3-5 points), Conclusion
# If state["feedback"] is not empty, incorporate the editor's feedback

def writer(state: ResearchState) -> dict:
    # YOUR CODE HERE
    return {"report": "", "attempts": state.get("attempts", 0) + 1}

# Test writer
test_state["findings"] = result["findings"]
result = writer(test_state)
print(f"Report length: {len(result['report'])} chars")
print(f"Attempts: {result['attempts']}")
if result["report"]:
    print(f"\n{result['report'][:300]}...")

In [ ]:
# Exercise 2d: Implement the editor node
# Score the report on: completeness, accuracy, clarity (each 0-1, average = quality_score)
# Return {"quality_score": float, "feedback": str}
# Use JSON mode

def editor(state: ResearchState) -> dict:
    # YOUR CODE HERE
    return {"quality_score": 0.0, "feedback": ""}

# Test editor
test_state["report"] = result["report"]
result = editor(test_state)
print(f"Quality score: {result['quality_score']:.2f}")
print(f"Feedback: {result['feedback']}")

## Step 3 · Assemble the Graph

In [ ]:
# Exercise 3: Build the StateGraph
# Nodes: planner, researcher, writer, editor
# Edges:
# - START → planner → researcher → writer → editor
# - editor → END if quality >= 0.80 or attempts >= 3
# - editor → writer otherwise (with feedback)

def route_after_editor(state: ResearchState) -> str:
    # YOUR CODE HERE
    return END

# YOUR CODE HERE — build and compile the graph
# builder = StateGraph(ResearchState)
# ... add nodes and edges ...
# graph = builder.compile()

print("Graph not yet assembled — implement the exercises above first")

In [ ]:
# Run the agent
# Uncomment after assembling the graph above

# initial_state = {
#     "topic": "impact of AI on healthcare in 2024",
#     "queries": [],
#     "findings": [],
#     "report": "",
#     "quality_score": 0.0,
#     "attempts": 0,
#     "feedback": "",
# }

# result = graph.invoke(initial_state)
# print(f"Final quality score: {result['quality_score']:.2f}")
# print(f"Total attempts: {result['attempts']}")
# print(f"\nFinal Report:")
# print(result["report"])

print("Uncomment and run after building the graph")

## Step 4 · Add Persistence (Stretch)

In [ ]:
# Stretch: Add MemorySaver so research can be resumed after interruption
# Also add interrupt_before=["writer"] so the human can review the findings
# before the writer node generates the report

from langgraph.checkpoint.memory import MemorySaver

# YOUR CODE HERE
# memory = MemorySaver()
# graph_with_memory = builder.compile(checkpointer=memory, interrupt_before=["writer"])

# config = {"configurable": {"thread_id": "research-001"}}
# Run until interrupt → show findings → approve → resume

print("MemorySaver stretch exercise — implement after core graph is working")